In [1]:
from pathlib import Path
import gcamreader
import os
import pandas as pd
import numpy as np
from utils import convert_to_mt
import plotly.graph_objects as go

In [2]:
def to_Mt(row):
    val, unit = row['value'], row['Units']
    if unit == 'Tg':
        return val
    elif unit == 'Gg':
        return val * 1e-3
    elif unit == 'MTC':
        return val * (44.009 / 12.011)
    else:
        raise ValueError(f"Unknown unit: {unit}")

# AR5 100-yr GWP
GWP_AR5 = {
    'CO2':    1,
    'CH4': 28,  # Methane
    'CH4_AGR': 28,  # Methane from Agriculture
    'CH4_AWB': 28,  # Methane from Agricultural Waste Burning
    'N2O': 265,  # Nitrous Oxide
    'N2O_AGR': 265,  # Nitrous Oxide from Agriculture
    'N2O_AWB': 265,  # Nitrous Oxide from Agricultural Waste Burning
    'HFC125': 3500,
    'HFC134a':1430,
    'HFC143a':4470,
    'HFC23':  14800,
    'HFC32':  675,
    'HFC43':  1500,
    'HFC227ea':3220,
    'HFC236fa':9810,
    'SF6':    23500,
    'C2F6':   12200,
    'CF4':    6630,
}

In [3]:
dfCO2Map = pd.read_csv("./extdata/gcamreport/CO2_tech_map.csv", skiprows=[0])
dfCO2Map.head()

,sector,subsector,technology,var1,var2,var3,var4,var5,var6,var7,var8,var9,unit_conv
0,airCO2,airCO2,airCO2,Emissions|CO2,Emissions|CO2|Other Capture and Removal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.666667
1,CO2 removal,dac,hightemp DAC NG,Emissions|CO2,Emissions|CO2|Other Capture and Removal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.666667
2,CO2 removal,dac,hightemp DAC elec,Emissions|CO2,Emissions|CO2|Other Capture and Removal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.666667
3,CO2 removal,dac,lowtemp DAC heatpump,Emissions|CO2,Emissions|CO2|Other Capture and Removal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.666667
4,agricultural energy use,mobile,refined liquids,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Demand,Emissions|CO2|Energy|Demand|AFOFI,Emissions|CO2|Energy|Demand|Residential and Co...,NaN,NaN,NaN,3.666667


In [4]:
dfNonCO2Map = pd.read_csv("./extdata/gcamreport/nonCO2_emissions_sector_map.csv", skiprows=[0])
dfNonCO2Map

,sector,subsector,ghg,var1,var2,var3,var4,var5,var6,var7,var8,unit_conv
0,agricultural energy use,NaN,BC,Emissions|BC,Emissions|BC|Energy,Emissions|BC|Energy|Demand,Emissions|BC|Energy|Demand|AFOFI,NaN,NaN,Emissions|BC|Energy|Demand|Residential and Com...,Emissions|BC|Energy and Industrial Processes,1.0
1,agricultural energy use,NaN,CH4,Emissions|CH4,Emissions|CH4|Energy,Emissions|BC|Energy|Demand,Emissions|CH4|Energy|Demand|AFOFI,NaN,NaN,Emissions|CH4|Energy|Demand|Residential and Co...,Emissions|CH4|Energy and Industrial Processes,1.0
2,agricultural energy use,NaN,CO,Emissions|CO,Emissions|CO|Energy,Emissions|BC|Energy|Demand,Emissions|CO|Energy|Demand|AFOFI,NaN,NaN,Emissions|CO|Energy|Demand|Residential and Com...,Emissions|CO|Energy and Industrial Processes,1.0
3,agricultural energy use,NaN,N2O,Emissions|N2O,Emissions|N2O|Energy,Emissions|N2O|Energy|Demand,Emissions|N2O|Energy|Demand|AFOFI,NaN,NaN,Emissions|N2O|Energy|Demand|Residential and Co...,Emissions|N2O|Energy and Industrial Processes,1000.0
4,agricultural energy use,NaN,NH3,Emissions|NH3,Emissions|NH3|Energy,Emissions|NH3|Energy|Demand,Emissions|NH3|Energy|Demand|AFOFI,NaN,NaN,Emissions|NH3|Energy|Demand|Residential and Co...,Emissions|NH3|Energy and Industrial Processes,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...
918,urban processes,NaN,OC,Emissions|OC,Emissions|OC|Other,NaN,NaN,NaN,NaN,NaN,NaN,1.0
919,urban processes,NaN,SO2_1,Emissions|Sulfur,Emissions|Sulfur|Other,NaN,NaN,NaN,NaN,NaN,NaN,1.0
920,urban processes,NaN,SO2_2,Emissions|Sulfur,Emissions|Sulfur|Other,NaN,NaN,NaN,NaN,NaN,NaN,1.0
921,urban processes,NaN,SO2_3,Emissions|Sulfur,Emissions|Sulfur|Other,NaN,NaN,NaN,NaN,NaN,NaN,1.0


In [114]:
proj_path = Path("/data/project/tae/gcam-core")
xml_path = proj_path / "input" / "gcamdata" / "xml"
db_path = proj_path / "output"

In [121]:
dbpath = "../output/"  # relative to current working directory
dbfile = "database_basexdb_korea_2035_20250717_4"
conn = gcamreader.LocalDBConn(dbpath, dbfile)
queries = gcamreader.parse_batch_query(os.path.join('..', 'output', 'queries','Main_queries.xml'))

Database scenarios: Current-Policy, Enhanced-Ambition


In [122]:
scenarios = list(conn.listScenariosInDB()['name'])
scenarios

['Current-Policy', 'Enhanced-Ambition']

In [123]:
scenarios.reverse()

In [124]:
for i, q in enumerate(queries):
    print(i, q.title)

0 primary energy consumption by region (avg fossil efficiency)
1 primary energy consumption by region (direct equivalent)
2 primary energy consumption with CCS by region (direct equivalent)
3 resource production
4 resource production by tech and vintage
5 resource supply curves
6 regional primary energy prices
7 elec gen by region (incl CHP)
8 elec gen by subsector
9 elec gen by gen tech
10 elec gen by gen tech and cooling tech
11 elec gen by gen tech and cooling tech and vintage
12 elec gen by gen tech and cooling tech (new)
13 elec energy input by subsector
14 elec energy input by elec gen tech
15 elec energy input by elec gen tech and cooling tech
16 elec prices by sector
17 elec gen costs by subsector
18 elec gen costs by tech
19 elec gen costs by cooling tech
20 elec share-weights by subsector
21 elec share-weights by tech
22 elec share-weights by cooling tech
23 elec td inputs and outputs
24 cogeneration by region
25 elec consumption by demand sector
26 elec sector water withdraw

In [125]:
q = queries[195]
print(q.title)
df = conn.runQuery(q, scenarios=scenarios, regions=['South Korea'])
df['scenario'] = df['scenario'].str.split(',').str[0]
df

ag production by crop type


,Units,scenario,region,sector,output,Year,value
0,EJ,Current-Policy,South Korea,biomass,biomass,2025,0.014514
1,EJ,Current-Policy,South Korea,biomass,biomass,2030,0.036085
2,EJ,Current-Policy,South Korea,biomass,biomass,2035,0.068044
3,EJ,Enhanced-Ambition,South Korea,biomass,biomass,2025,0.014513
4,EJ,Enhanced-Ambition,South Korea,biomass,biomass,2030,0.069095
...,...,...,...,...,...,...,...
289,billion m3,Enhanced-Ambition,South Korea,Forest,Forest,2015,0.004540
290,billion m3,Enhanced-Ambition,South Korea,Forest,Forest,2020,0.004670
291,billion m3,Enhanced-Ambition,South Korea,Forest,Forest,2025,0.004702
292,billion m3,Enhanced-Ambition,South Korea,Forest,Forest,2030,0.004733


In [126]:
df['sector'].unique()

array(['biomass', 'Corn', 'FiberCrop', 'FodderGrass', 'Fruits', 'Legumes',
       'MiscCrop', 'NutsSeeds', 'OilCrop', 'OtherGrain', 'Pasture',
       'Rice', 'RootTuber', 'Soybean', 'Vegetables', 'Wheat', 'Forest'],
      dtype=object)

In [127]:
df[(df['sector'] == 'Rice')]

,Units,scenario,region,sector,output,Year,value
96,Mt,Current-Policy,South Korea,Rice,Rice,1975,6.821400
97,Mt,Current-Policy,South Korea,Rice,Rice,1990,7.726534
98,Mt,Current-Policy,South Korea,Rice,Rice,2005,6.318193
99,Mt,Current-Policy,South Korea,Rice,Rice,2010,5.960649
100,Mt,Current-Policy,South Korea,Rice,Rice,2015,5.590278
101,Mt,Current-Policy,South Korea,Rice,Rice,2020,5.835995
102,Mt,Current-Policy,South Korea,Rice,Rice,2025,6.220490
103,Mt,Current-Policy,South Korea,Rice,Rice,2030,6.701642
104,Mt,Current-Policy,South Korea,Rice,Rice,2035,6.865523
231,Mt,Enhanced-Ambition,South Korea,Rice,Rice,1975,6.821400


In [128]:
q = queries[262]
print(q.title)
dfCO2 = conn.runQuery(q, scenarios=scenarios, regions=['South Korea'])
dfCO2['scenario'] = dfCO2['scenario'].str.split(',').str[0]
dfCO2['sector'] = dfCO2['sector'].str.replace(r'_d(?:[1-9]|10)$', '', regex=True)
dfCO2['GHG'] = 'CO2'

CO2 emissions by sector (no bio) (excluding resource production)


In [129]:
q = queries[272]
print(q.title)
dfNonCO2 = conn.runQuery(q, scenarios=scenarios, regions=['South Korea'])
dfNonCO2['scenario'] = dfNonCO2['scenario'].str.split(',').str[0]
dfNonCO2['sector'] = dfNonCO2['sector'].str.replace(r'_d(?:[1-9]|10)$', '', regex=True)
dfNonCO2 = dfNonCO2[(dfNonCO2['GHG'].isin(GWP_AR5.keys()))]
dfNonCO2

nonCO2 emissions by subsector (excluding resource production)


,Units,scenario,region,sector,subsector,GHG,Year,value
0,Gg,Current-Policy,South Korea,comm cooling,electricity,HFC125,2005,0.253100
1,Gg,Current-Policy,South Korea,comm cooling,electricity,HFC125,2010,0.629604
2,Gg,Current-Policy,South Korea,comm cooling,electricity,HFC125,2015,0.717899
3,Gg,Current-Policy,South Korea,comm cooling,electricity,HFC125,2020,0.796811
4,Gg,Current-Policy,South Korea,comm cooling,electricity,HFC125,2025,0.788475
...,...,...,...,...,...,...,...,...
23011,Tg,Enhanced-Ambition,South Korea,waste biomass for paper,biomass,N2O,2015,0.000123
23012,Tg,Enhanced-Ambition,South Korea,waste biomass for paper,biomass,N2O,2020,0.000124
23013,Tg,Enhanced-Ambition,South Korea,waste biomass for paper,biomass,N2O,2025,0.000123
23014,Tg,Enhanced-Ambition,South Korea,waste biomass for paper,biomass,N2O,2030,0.000121


In [130]:
mask1 = ((dfNonCO2['sector'] == 'UnmanagedLand') & (dfNonCO2['subsector'].isin(['ForestFire', 'GrasslandFires'])))
mask2 = ((dfNonCO2['sector'] == 'urban processes') & (dfNonCO2['subsector'].isin(['landfills', 'wastewater', 'waste_incineration'])))

dfNonCO2_1 = dfNonCO2[~(mask1 | mask2)]
dfNonCO2_2 = dfNonCO2[mask1 | mask2]

print(dfNonCO2_1.shape, dfNonCO2_2.shape)

(6144, 8) (90, 8)


In [131]:
for sec in dfCO2['sector'].unique():
    if sec not in dfCO2Map['sector'].unique():
        print(sec)

electricity


In [132]:
for sec in dfNonCO2['sector'].unique():
    if sec not in dfNonCO2Map['sector'].unique():
        print(sec)

In [133]:
dfNonCO2[(dfNonCO2['sector'] == 'chemical feedstocks')]

,Units,scenario,region,sector,subsector,GHG,Year,value


In [134]:
dfCO2Map.head()

,sector,subsector,technology,var1,var2,var3,var4,var5,var6,var7,var8,var9,unit_conv
0,airCO2,airCO2,airCO2,Emissions|CO2,Emissions|CO2|Other Capture and Removal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.666667
1,CO2 removal,dac,hightemp DAC NG,Emissions|CO2,Emissions|CO2|Other Capture and Removal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.666667
2,CO2 removal,dac,hightemp DAC elec,Emissions|CO2,Emissions|CO2|Other Capture and Removal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.666667
3,CO2 removal,dac,lowtemp DAC heatpump,Emissions|CO2,Emissions|CO2|Other Capture and Removal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.666667
4,agricultural energy use,mobile,refined liquids,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Demand,Emissions|CO2|Energy|Demand|AFOFI,Emissions|CO2|Energy|Demand|Residential and Co...,NaN,NaN,NaN,3.666667


In [135]:
dfCO2Sec = dfCO2.merge(dfCO2Map[['sector', 'var1', 'var2', 'var3', 'var4', 'var5']].drop_duplicates(), on=['sector'], how='left')
dfCO2Sec.head()

,Units,scenario,region,sector,Year,value,GHG,var1,var2,var3,var4,var5
0,MTC,Current-Policy,South Korea,H2 central production,2020,0.002978,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen
1,MTC,Current-Policy,South Korea,H2 central production,2025,0.005733,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen
2,MTC,Current-Policy,South Korea,H2 central production,2030,0.002513,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen
3,MTC,Current-Policy,South Korea,H2 central production,2035,0.005235,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen
4,MTC,Current-Policy,South Korea,H2 wholesale dispensing,2020,0.002867,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen


In [136]:
dfNonCO2_1Sec = dfNonCO2_1.merge(dfNonCO2Map[(dfNonCO2Map['subsector'].isna())][['sector', 'ghg', 'var1', 'var2', 'var3', 'var4', 'var5']].drop_duplicates(), left_on=['sector', 'GHG'], right_on=['sector', 'ghg'], how='left')
dfNonCO2_1Sec.head()

,Units,scenario,region,sector,subsector,GHG,Year,value,ghg,var1,var2,var3,var4,var5
0,Gg,Current-Policy,South Korea,comm cooling,electricity,HFC125,2005,0.253100,HFC125,Emissions|HFC,Emissions|HFC|HFC125,Emissions|F-Gases,NaN,NaN
1,Gg,Current-Policy,South Korea,comm cooling,electricity,HFC125,2010,0.629604,HFC125,Emissions|HFC,Emissions|HFC|HFC125,Emissions|F-Gases,NaN,NaN
2,Gg,Current-Policy,South Korea,comm cooling,electricity,HFC125,2015,0.717899,HFC125,Emissions|HFC,Emissions|HFC|HFC125,Emissions|F-Gases,NaN,NaN
3,Gg,Current-Policy,South Korea,comm cooling,electricity,HFC125,2020,0.796811,HFC125,Emissions|HFC,Emissions|HFC|HFC125,Emissions|F-Gases,NaN,NaN
4,Gg,Current-Policy,South Korea,comm cooling,electricity,HFC125,2025,0.788475,HFC125,Emissions|HFC,Emissions|HFC|HFC125,Emissions|F-Gases,NaN,NaN


In [137]:
dfNonCO2_2Sec = dfNonCO2_2.merge(dfNonCO2Map[['sector', 'subsector', 'ghg', 'var1', 'var2', 'var3', 'var4', 'var5']].drop_duplicates(), left_on=['sector', 'subsector', 'GHG'], right_on=['sector', 'subsector', 'ghg'], how='left')
dfNonCO2_2Sec.head()

,Units,scenario,region,sector,subsector,GHG,Year,value,ghg,var1,var2,var3,var4,var5
0,Tg,Current-Policy,South Korea,urban processes,landfills,CH4,1975,0.013917,CH4,Emissions|CH4,Emissions|CH4|Waste,NaN,NaN,NaN
1,Tg,Current-Policy,South Korea,urban processes,landfills,CH4,1990,0.031836,CH4,Emissions|CH4,Emissions|CH4|Waste,NaN,NaN,NaN
2,Tg,Current-Policy,South Korea,urban processes,landfills,CH4,2005,0.042066,CH4,Emissions|CH4,Emissions|CH4|Waste,NaN,NaN,NaN
3,Tg,Current-Policy,South Korea,urban processes,landfills,CH4,2010,0.021372,CH4,Emissions|CH4,Emissions|CH4|Waste,NaN,NaN,NaN
4,Tg,Current-Policy,South Korea,urban processes,landfills,CH4,2015,0.021774,CH4,Emissions|CH4,Emissions|CH4|Waste,NaN,NaN,NaN


In [138]:
dfNonCO2_1.shape, dfNonCO2_1Sec.shape

((6144, 8), (6144, 14))

In [139]:
dfNonCO2_2.shape, dfNonCO2_2Sec.shape

((90, 8), (90, 14))

In [140]:
dfNonCO2Sec = pd.concat([dfNonCO2_1Sec, dfNonCO2_2Sec])
dfNonCO2Sec

,Units,scenario,region,sector,subsector,GHG,Year,value,ghg,var1,var2,var3,var4,var5
0,Gg,Current-Policy,South Korea,comm cooling,electricity,HFC125,2005,0.253100,HFC125,Emissions|HFC,Emissions|HFC|HFC125,Emissions|F-Gases,NaN,NaN
1,Gg,Current-Policy,South Korea,comm cooling,electricity,HFC125,2010,0.629604,HFC125,Emissions|HFC,Emissions|HFC|HFC125,Emissions|F-Gases,NaN,NaN
2,Gg,Current-Policy,South Korea,comm cooling,electricity,HFC125,2015,0.717899,HFC125,Emissions|HFC,Emissions|HFC|HFC125,Emissions|F-Gases,NaN,NaN
3,Gg,Current-Policy,South Korea,comm cooling,electricity,HFC125,2020,0.796811,HFC125,Emissions|HFC,Emissions|HFC|HFC125,Emissions|F-Gases,NaN,NaN
4,Gg,Current-Policy,South Korea,comm cooling,electricity,HFC125,2025,0.788475,HFC125,Emissions|HFC,Emissions|HFC|HFC125,Emissions|F-Gases,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,Tg,Enhanced-Ambition,South Korea,urban processes,wastewater,N2O,2015,0.003289,N2O,Emissions|N2O,Emissions|N2O|Waste,NaN,NaN,NaN
86,Tg,Enhanced-Ambition,South Korea,urban processes,wastewater,N2O,2020,0.003340,N2O,Emissions|N2O,Emissions|N2O|Waste,NaN,NaN,NaN
87,Tg,Enhanced-Ambition,South Korea,urban processes,wastewater,N2O,2025,0.003312,N2O,Emissions|N2O,Emissions|N2O|Waste,NaN,NaN,NaN
88,Tg,Enhanced-Ambition,South Korea,urban processes,wastewater,N2O,2030,0.002672,N2O,Emissions|N2O,Emissions|N2O|Waste,NaN,NaN,NaN


In [141]:
dfCO2['sector'].unique()

array(['H2 central production', 'H2 wholesale dispensing',
       'agricultural energy use', 'ammonia', 'backup_electricity',
       'cement', 'chemical energy use', 'chemical feedstocks',
       'comm cooling', 'comm heating', 'comm others',
       'construction energy use', 'construction feedstocks',
       'delivered biomass', 'delivered gas', 'desalinated water',
       'elec_coal (IGCC CCS)', 'elec_coal (conv pul CCS)',
       'elec_gas (CC CCS)', 'electricity', 'gas pipeline',
       'gas processing', 'iron and steel', 'mining energy use',
       'other industrial energy use', 'other industrial feedstocks',
       'process heat cement', 'process heat food processing',
       'process heat paper', 'refined liquids enduse',
       'refined liquids industrial', 'refining', 'resid heating coal',
       'resid heating modern', 'resid others coal', 'resid others modern',
       'trn_aviation_intl', 'trn_freight', 'trn_freight_road', 'trn_pass',
       'trn_pass_road', 'trn_pass_road_LD

In [142]:
def cat_sec(row):
    if row['sector'] == 'electricity':
        return 'Electricity'
    elif row['sector'] == 'agricultural energy use':
        return 'Agriculture'
    elif row['var2'].endswith('Other Capture and Removal'):
        return 'DAC'
    elif row['sector'] == 'cement':
        return 'Industry'
    elif row['sector'] == 'desalinated water':
        return 'Buildings'
    # elif row['var5'].endswith('Hydrogen'):
    #     return 'Hydrogen'
    elif row['var5'].endswith('Industry'):
        return 'Industry'
    elif row['var5'].endswith('Electricity'):
        return "Electricity"
    elif row['var5'].endswith('Residential and Commercial'):
        return "Buildings"
    elif row['var5'].endswith('Transportation'):
        return 'Transportation'
    elif row['sector'] in ['delivered biomass', 'delivered gas', 'gas pipeline', 'gas processing', 'refined liquids enduse', 'refined liquids industrial', 'refining', 'wholesale gas']:
        return 'Industry'
    elif row['var5'].endswith('AFOFI'):
        return 'Industry'
    else:
        print(row['sector'])
        return "Others"

In [143]:
dfCO2Sec['sec'] = dfCO2Sec.apply(cat_sec, axis=1)

H2 central production
H2 central production
H2 central production
H2 central production
H2 wholesale dispensing
H2 wholesale dispensing
H2 wholesale dispensing
H2 wholesale dispensing
H2 central production
H2 central production
H2 central production
H2 central production
H2 wholesale dispensing
H2 wholesale dispensing
H2 wholesale dispensing
H2 wholesale dispensing


In [144]:
dfNonCO2Sec[~(dfNonCO2Sec['var4'].isna())]['sector'].unique()

array(['industrial processes', 'Beef', 'Corn', 'Dairy', 'FiberCrop',
       'Fruits', 'H2 central production', 'Legumes', 'MiscCrop',
       'NutsSeeds', 'OilCrop', 'OtherGrain', 'Pork', 'Poultry', 'Rice',
       'RootTuber', 'SheepGoat', 'Soybean', 'UnmanagedLand', 'Vegetables',
       'Wheat', 'agricultural energy use', 'ammonia',
       'backup_electricity', 'biomass', 'chemical energy use',
       'comm cooling', 'comm heating', 'comm others',
       'construction energy use', 'electricity', 'mining energy use',
       'other industrial energy use', 'process heat cement',
       'process heat food processing', 'process heat paper', 'refining',
       'resid heating TradBio', 'resid heating coal',
       'resid heating modern', 'resid others TradBio',
       'resid others coal', 'resid others modern', 'trn_aviation_intl',
       'trn_freight', 'trn_freight_road', 'trn_pass', 'trn_pass_road',
       'trn_pass_road_LDV', 'trn_pass_road_LDV_4W', 'trn_shipping_intl'],
      dtype=object

In [145]:
dfNonCO2Sec[(dfNonCO2Sec['sector'] == 'urban processes') & ~(dfNonCO2Sec['GHG'].isin(['HFC125', 'HFC134a', 'HFC143a', 'HFC23', 'HFC32', 'HFC43', 'HFC227ea', 'HFC236fa', 'SF6', 'C2F6', 'CF4']))]['var2'].unique()

array(['Emissions|CH4|Waste', 'Emissions|N2O|Waste'], dtype=object)

In [146]:
dfNonCO2Sec['sector'].unique()

array(['comm cooling', 'electricity_net_ownuse', 'industrial processes',
       'resid cooling modern', 'urban processes', 'Beef', 'Corn', 'Dairy',
       'FiberCrop', 'FodderGrass', 'Fruits', 'H2 central production',
       'Legumes', 'MiscCrop', 'NutsSeeds', 'OilCrop', 'OtherGrain',
       'Pork', 'Poultry', 'Rice', 'RootTuber', 'SheepGoat', 'Soybean',
       'UnmanagedLand', 'Vegetables', 'Wheat', 'agricultural energy use',
       'ammonia', 'backup_electricity', 'biomass', 'chemical energy use',
       'comm heating', 'comm others', 'construction energy use',
       'electricity', 'iron and steel', 'mining energy use',
       'other industrial energy use', 'process heat cement',
       'process heat food processing', 'process heat paper', 'refining',
       'resid heating TradBio', 'resid heating coal',
       'resid heating modern', 'resid others TradBio',
       'resid others coal', 'resid others modern', 'trn_aviation_intl',
       'trn_freight', 'trn_freight_road', 'trn_pass', 

In [147]:
def cat_sec_nonco2(row):
    if row['GHG'] in ['HFC125', 'HFC134a', 'HFC143a', 'HFC23', 'HFC32', 'HFC43', 'HFC227ea', 'HFC236fa', 'SF6', 'C2F6', 'CF4']:
        return 'F-Gases'
    elif row['sector'] == 'agricultural energy use':
        return "Agriculture"
    elif row['var2'].endswith("Waste"):
        return "Waste"
    elif (row['var2'].endswith("AFOLU")) and ((row['var3'].endswith("Agriculture")) or (row['var3'].endswith('Agricultural Waste Burning'))):
        return 'Agriculture'
    elif (row['var2'].endswith("AFOLU")):
        return 'Others'
    elif (row['var2'].endswith('Industrial Processes')) or (row['sector'] == 'industrial processes'):
        return 'Industry'
    elif (row['var2'].endswith('Waste')) or (row['sector'] == 'urban processes'):
        return 'Others'#'Waste'
    elif row['var4'].endswith('Electricity'):
        return 'Electricity'
    elif (row['var2'].endswith('Industrial Processes')) or (row['var4'].endswith('Industry')):
        return 'Industry'
    elif row['var4'].endswith('Transportation'):
        return 'Transportation'
    # elif row['var4'].endswith('Hydrogen'):
    #     return 'Hydrogen' 
    elif row['var4'].endswith('Residential and Commercial'):
        return "Buildings"
    elif (row['var4'].endswith('AFOFI')) or (row['var4'].endswith('Heat')) or (row['var4'].endswith('Liquids')):
        return 'Industry'
    else:
        print(row['sector'])
        return "Others"

In [148]:
dfNonCO2Sec['sec'] = dfNonCO2Sec.apply(cat_sec_nonco2, axis=1)

H2 central production
H2 central production
H2 central production
H2 central production
trn_aviation_intl
trn_aviation_intl
trn_aviation_intl
trn_aviation_intl
trn_aviation_intl
trn_aviation_intl
trn_aviation_intl
trn_aviation_intl
trn_aviation_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
H2 central production
H2 central production
H2 central production
H2 central production
trn_aviation_intl
trn_aviation_intl
trn_aviation_intl
trn_aviation_intl
trn_aviation_intl
trn_aviation_intl
trn_aviation_intl
trn_aviation_intl
trn_aviation_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_i

In [149]:
dfGHGSec = pd.concat([dfCO2Sec, dfNonCO2Sec])
dfGHGSec

,Units,scenario,region,sector,Year,value,GHG,var1,var2,var3,var4,var5,sec,subsector,ghg
0,MTC,Current-Policy,South Korea,H2 central production,2020,0.002978,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen,Others,NaN,NaN
1,MTC,Current-Policy,South Korea,H2 central production,2025,0.005733,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen,Others,NaN,NaN
2,MTC,Current-Policy,South Korea,H2 central production,2030,0.002513,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen,Others,NaN,NaN
3,MTC,Current-Policy,South Korea,H2 central production,2035,0.005235,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen,Others,NaN,NaN
4,MTC,Current-Policy,South Korea,H2 wholesale dispensing,2020,0.002867,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen,Others,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,Tg,Enhanced-Ambition,South Korea,urban processes,2015,0.003289,N2O,Emissions|N2O,Emissions|N2O|Waste,NaN,NaN,NaN,Waste,wastewater,N2O
86,Tg,Enhanced-Ambition,South Korea,urban processes,2020,0.003340,N2O,Emissions|N2O,Emissions|N2O|Waste,NaN,NaN,NaN,Waste,wastewater,N2O
87,Tg,Enhanced-Ambition,South Korea,urban processes,2025,0.003312,N2O,Emissions|N2O,Emissions|N2O|Waste,NaN,NaN,NaN,Waste,wastewater,N2O
88,Tg,Enhanced-Ambition,South Korea,urban processes,2030,0.002672,N2O,Emissions|N2O,Emissions|N2O|Waste,NaN,NaN,NaN,Waste,wastewater,N2O


In [150]:
dfGHGSec['sec'].unique()

array(['Others', 'Agriculture', 'Industry', 'Electricity', 'Buildings',
       'Transportation', 'DAC', 'F-Gases', 'Waste'], dtype=object)

In [151]:
dfGHGSec[(dfGHGSec['sec'] == 'Agriculture')]['var3'].unique()

array(['Emissions|CO2|Energy', 'Emissions|CH4|AFOLU|Agriculture',
       'Emissions|N2O|AFOLU|Agriculture',
       'Emissions|CH4|AFOLU|Agricultural Waste Burning',
       'Emissions|N2O|AFOLU|Agricultural Waste Burning',
       'Emissions|BC|Energy|Demand', 'Emissions|N2O|Energy|Demand'],
      dtype=object)

In [152]:
dfGHGSec['var3'].unique()

array(['Emissions|CO2|Energy', 'Emissions|CO2|Industrial Processes', nan,
       'Emissions|F-Gases', 'Emissions|CH4|AFOLU|Agriculture',
       'Emissions|N2O|AFOLU|Agriculture',
       'Emissions|CH4|AFOLU|Agricultural Waste Burning',
       'Emissions|N2O|AFOLU|Agricultural Waste Burning',
       'Emissions|N2O|AFOLU|Land', 'Emissions|CH4|Energy|Supply',
       'Emissions|CH4|AFOLU|Land', 'Emissions|BC|Energy|Demand',
       'Emissions|N2O|Energy|Demand',
       'Emissions|CH4|Industrial Processes|Chemicals',
       'Emissions|N2O|Industrial Processes|Chemicals',
       'Emissions|N2O|Energy|Supply', 'Emissions|CH4|Energy|Demand',
       'Emissions|CH4|Industrial Processes|Iron and Steel',
       'Emissions|N2O|Industrial Processes|Iron and Steel',
       'Emissions|CH4|Industrial Processes|Pulp and Paper',
       'Emissions|N2O|Industrial Processes|Pulp and Paper'], dtype=object)

In [153]:
dfGHGSec['emiss(MT)'] = dfGHGSec.apply(convert_to_mt, axis=1)
dfGHGSec['gwpAr5'] = dfGHGSec['GHG'].apply(lambda gas: GWP_AR5[gas] if gas in GWP_AR5 else np.nan)
dfGHGSec['MTCO2eq'] = dfGHGSec['emiss(MT)'] * dfGHGSec['gwpAr5']
dfGHGSec

,Units,scenario,region,sector,Year,value,GHG,var1,var2,var3,var4,var5,sec,subsector,ghg,emiss(MT),gwpAr5,MTCO2eq
0,MTC,Current-Policy,South Korea,H2 central production,2020,0.002978,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen,Others,NaN,NaN,0.010918,1,0.010918
1,MTC,Current-Policy,South Korea,H2 central production,2025,0.005733,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen,Others,NaN,NaN,0.021019,1,0.021019
2,MTC,Current-Policy,South Korea,H2 central production,2030,0.002513,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen,Others,NaN,NaN,0.009215,1,0.009215
3,MTC,Current-Policy,South Korea,H2 central production,2035,0.005235,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen,Others,NaN,NaN,0.019197,1,0.019197
4,MTC,Current-Policy,South Korea,H2 wholesale dispensing,2020,0.002867,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen,Others,NaN,NaN,0.010511,1,0.010511
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,Tg,Enhanced-Ambition,South Korea,urban processes,2015,0.003289,N2O,Emissions|N2O,Emissions|N2O|Waste,NaN,NaN,NaN,Waste,wastewater,N2O,0.003289,265,0.871601
86,Tg,Enhanced-Ambition,South Korea,urban processes,2020,0.003340,N2O,Emissions|N2O,Emissions|N2O|Waste,NaN,NaN,NaN,Waste,wastewater,N2O,0.003340,265,0.885222
87,Tg,Enhanced-Ambition,South Korea,urban processes,2025,0.003312,N2O,Emissions|N2O,Emissions|N2O|Waste,NaN,NaN,NaN,Waste,wastewater,N2O,0.003312,265,0.877770
88,Tg,Enhanced-Ambition,South Korea,urban processes,2030,0.002672,N2O,Emissions|N2O,Emissions|N2O|Waste,NaN,NaN,NaN,Waste,wastewater,N2O,0.002672,265,0.708207


In [154]:
dfGHGSec[(dfGHGSec['sector'] == 'Rice') & (dfGHGSec['Year'] == 2035)].groupby(['scenario', 'Year', 'subsector'])['MTCO2eq'].sum()

scenario           Year  subsector 
Current-Policy     2035  Rice_Korea    4.533750
Enhanced-Ambition  2035  Rice_Korea    3.748272
Name: MTCO2eq, dtype: float64

In [155]:
dfGHGSec[(dfGHGSec['sec'].isin(['Agriculture']))& (dfGHGSec['Year'] == 2035)].groupby(['scenario', 'Year', 'sector'])['MTCO2eq'].sum()

scenario           Year  sector                 
Current-Policy     2035  Beef                       4.190004
                         Corn                       0.110991
                         Dairy                      1.624005
                         FiberCrop                  0.001851
                         Fruits                     0.473455
                         Legumes                    0.056332
                         MiscCrop                   0.065252
                         NutsSeeds                  0.148092
                         OilCrop                    0.345713
                         OtherGrain                 0.170496
                         Pork                       3.980833
                         Poultry                    0.217733
                         Rice                       4.533750
                         RootTuber                  0.153296
                         SheepGoat                  0.235726
                         Soybean    

In [156]:
dfGHGSec[(dfGHGSec['Year'].isin([2020, 2035])) & (dfGHGSec['sec'] == 'Waste')].groupby(['scenario', 'Year', 'subsector'])['MTCO2eq'].sum()

scenario           Year  subsector         
Current-Policy     2020  landfills              0.455641
                         waste_incineration     0.029051
                         wastewater            14.878558
                   2035  landfills              0.499192
                         waste_incineration     0.024775
                         wastewater            12.641603
Enhanced-Ambition  2020  landfills              0.455641
                         waste_incineration     0.029051
                         wastewater            14.878558
                   2035  landfills              0.313642
                         waste_incineration     0.020612
                         wastewater            10.518999
Name: MTCO2eq, dtype: float64

In [157]:
dfGHGSec.groupby(['scenario', 'Year'])['MTCO2eq'].sum()

scenario           Year
Current-Policy     1975     54.168502
                   1990    296.523004
                   2005    601.039565
                   2010    698.621279
                   2015    747.006066
                   2020    737.456948
                   2025    703.591905
                   2030    632.278461
                   2035    567.181000
Enhanced-Ambition  1975     54.168502
                   1990    296.523004
                   2005    601.039565
                   2010    698.621279
                   2015    747.006066
                   2020    737.456948
                   2025    703.489046
                   2030    527.933570
                   2035    385.118292
Name: MTCO2eq, dtype: float64

In [158]:
dfGHGSec[(dfGHGSec['sector'].str.contains('H2')) & (~dfGHGSec['sector'].isin(['trn_aviation_intl', 'trn_shipping_intl'])) & (dfGHGSec['Year'] == 2035)]#['var4'].unique()

,Units,scenario,region,sector,Year,value,GHG,var1,var2,var3,var4,var5,sec,subsector,ghg,emiss(MT),gwpAr5,MTCO2eq
3,MTC,Current-Policy,South Korea,H2 central production,2035,5.235466e-03,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen,Others,NaN,NaN,1.919671e-02,1,0.019197
7,MTC,Current-Policy,South Korea,H2 wholesale dispensing,2035,5.679309e-02,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen,Others,NaN,NaN,2.082413e-01,1,0.208241
594,MTC,Enhanced-Ambition,South Korea,H2 central production,2035,1.206967e-03,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen,Others,NaN,NaN,4.425546e-03,1,0.004426
598,MTC,Enhanced-Ambition,South Korea,H2 wholesale dispensing,2035,7.961302e-03,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen,Others,NaN,NaN,2.919144e-02,1,0.029191
1096,Tg,Current-Policy,South Korea,H2 central production,2035,2.504138e-06,CH4,Emissions|CH4,Emissions|CH4|Energy,Emissions|CH4|Energy|Supply,Emissions|CH4|Energy|Supply|Hydrogen,NaN,Others,biomass,CH4,2.504138e-06,28,0.000070
3704,Tg,Enhanced-Ambition,South Korea,H2 central production,2035,7.843040e-07,CH4,Emissions|CH4,Emissions|CH4|Energy,Emissions|CH4|Energy|Supply,Emissions|CH4|Energy|Supply|Hydrogen,NaN,Others,biomass,CH4,7.843040e-07,28,0.000022


In [159]:
dfGHGSec[(dfGHGSec['sec'] == 'Others') & (~dfGHGSec['sector'].isin(['trn_aviation_intl', 'trn_shipping_intl'])) & (dfGHGSec['Year'] == 2035)]#['var4'].unique()

,Units,scenario,region,sector,Year,value,GHG,var1,var2,var3,var4,var5,sec,subsector,ghg,emiss(MT),gwpAr5,MTCO2eq
3,MTC,Current-Policy,South Korea,H2 central production,2035,5.235466e-03,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen,Others,NaN,NaN,1.919671e-02,1,0.019197
7,MTC,Current-Policy,South Korea,H2 wholesale dispensing,2035,5.679309e-02,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen,Others,NaN,NaN,2.082413e-01,1,0.208241
594,MTC,Enhanced-Ambition,South Korea,H2 central production,2035,1.206967e-03,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen,Others,NaN,NaN,4.425546e-03,1,0.004426
598,MTC,Enhanced-Ambition,South Korea,H2 wholesale dispensing,2035,7.961302e-03,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen,Others,NaN,NaN,2.919144e-02,1,0.029191
1038,Tg,Current-Policy,South Korea,FodderGrass,2035,1.201770e-04,N2O_AGR,Emissions|N2O,Emissions|N2O|AFOLU,Emissions|N2O|AFOLU|Land,NaN,NaN,Others,FodderGrass_Korea,N2O_AGR,1.201770e-04,265,0.031847
1096,Tg,Current-Policy,South Korea,H2 central production,2035,2.504138e-06,CH4,Emissions|CH4,Emissions|CH4|Energy,Emissions|CH4|Energy|Supply,Emissions|CH4|Energy|Supply|Hydrogen,NaN,Others,biomass,CH4,2.504138e-06,28,0.000070
1447,Tg,Current-Policy,South Korea,UnmanagedLand,2035,3.441660e-03,CH4,Emissions|CH4,Emissions|CH4|AFOLU,Emissions|CH4|AFOLU|Land,Emissions|CH4|AFOLU|Land|Other,NaN,Others,Deforest_Korea,CH4,3.441660e-03,28,0.096366
1453,Tg,Current-Policy,South Korea,UnmanagedLand,2035,1.357660e-04,N2O,Emissions|N2O,Emissions|N2O|AFOLU,Emissions|N2O|AFOLU|Land,Emissions|N2O|AFOLU|Land|Other,NaN,Others,Deforest_Korea,N2O,1.357660e-04,265,0.035978
1462,Tg,Current-Policy,South Korea,UnmanagedLand,2035,9.324299e-05,CH4,Emissions|CH4,Emissions|CH4|AFOLU,Emissions|CH4|AFOLU|Land,Emissions|CH4|AFOLU|Land|Other,NaN,Others,ForestFire_Korea,CH4,9.324299e-05,28,0.002611
1471,Tg,Current-Policy,South Korea,UnmanagedLand,2035,4.440178e-06,N2O,Emissions|N2O,Emissions|N2O|AFOLU,Emissions|N2O|AFOLU|Land,Emissions|N2O|AFOLU|Land|Other,NaN,Others,ForestFire_Korea,N2O,4.440178e-06,265,0.001177


In [160]:
dfGHGSecDiff = dfGHGSec[(dfGHGSec['Year'].isin([2020, 2035]) & (~dfGHGSec['sector'].isin(['trn_aviation_intl', 'trn_shipping_intl']))) & (dfGHGSec['scenario'].isin(['Current-Policy', 'Enhanced-Ambition', 'Enhanced-Ambition-Bld']))].groupby(['scenario', 'Year', 'sec'])['MTCO2eq'].sum().reset_index().pivot(index=['scenario', 'sec'], columns=['Year'], values='MTCO2eq')
dfGHGSecDiff

Year                                    2020        2035
scenario          sec                                   
Current-Policy    Agriculture      17.823570   18.639860
                  Buildings        51.780272   48.850079
                  Electricity     243.949984  118.341128
                  F-Gases          42.544553   35.632696
                  Industry        221.311343  205.259027
                  Others            0.106952    0.402501
                  Transportation  118.196168  100.058581
                  Waste            15.363250   13.165570
Enhanced-Ambition Agriculture      17.823570   16.840806
                  Buildings        51.780272   36.088072
                  DAC                    NaN   -5.919833
                  Electricity     243.949984   51.467631
                  F-Gases          42.544553   26.330346
                  Industry        221.311343  133.396791
                  Others            0.106952    0.267494
                  Transportation  118.196168   89.348627
                  Waste            15.363250   10.853253

In [161]:
dfGHGSec[(dfGHGSec['sec'] == 'Others') & (~dfGHGSec['sector'].isin(['trn_aviation_intl', 'trn_shipping_intl']))].groupby(['Year','scenario', 'var2'])['MTCO2eq'].sum()

Year  scenario           var2                                         
1975  Current-Policy     Emissions|CH4|AFOLU                              5.970117e-03
                         Emissions|N2O|AFOLU                              8.032164e-02
      Enhanced-Ambition  Emissions|CH4|AFOLU                              5.970117e-03
                         Emissions|N2O|AFOLU                              8.032164e-02
1990  Current-Policy     Emissions|CH4|AFOLU                              5.603996e-03
                         Emissions|N2O|AFOLU                              2.636275e-01
      Enhanced-Ambition  Emissions|CH4|AFOLU                              5.603996e-03
                         Emissions|N2O|AFOLU                              2.636275e-01
2005  Current-Policy     Emissions|CH4|AFOLU                              6.253142e-03
                         Emissions|N2O|AFOLU                              3.976664e-02
      Enhanced-Ambition  Emissions|CH4|AFOLU               

In [162]:
dfGHGSec[(~dfGHGSec['sector'].isin(['trn_aviation_intl', 'trn_shipping_intl']))].groupby(['Year', 'scenario'])['MTCO2eq'].sum()

Year  scenario         
1975  Current-Policy        53.918215
      Enhanced-Ambition     53.918215
1990  Current-Policy       291.119683
      Enhanced-Ambition    291.119683
2005  Current-Policy       569.489759
      Enhanced-Ambition    569.489759
2010  Current-Policy       671.273365
      Enhanced-Ambition    671.273365
2015  Current-Policy       720.820835
      Enhanced-Ambition    720.820835
2020  Current-Policy       711.076092
      Enhanced-Ambition    711.076092
2025  Current-Policy       676.686438
      Enhanced-Ambition    676.583579
2030  Current-Policy       605.683078
      Enhanced-Ambition    501.731019
2035  Current-Policy       540.349442
      Enhanced-Ambition    358.673187
Name: MTCO2eq, dtype: float64

In [163]:
dfGHGSecDiff.fillna(0, inplace=True)
dfGHGSecDiff['Diff'] = dfGHGSecDiff[2035] - dfGHGSecDiff[2020]
dfGHGSecDiff

Year                                    2020        2035        Diff
scenario          sec                                               
Current-Policy    Agriculture      17.823570   18.639860    0.816289
                  Buildings        51.780272   48.850079   -2.930193
                  Electricity     243.949984  118.341128 -125.608855
                  F-Gases          42.544553   35.632696   -6.911857
                  Industry        221.311343  205.259027  -16.052316
                  Others            0.106952    0.402501    0.295549
                  Transportation  118.196168  100.058581  -18.137587
                  Waste            15.363250   13.165570   -2.197680
Enhanced-Ambition Agriculture      17.823570   16.840806   -0.982765
                  Buildings        51.780272   36.088072  -15.692200
                  DAC               0.000000   -5.919833   -5.919833
                  Electricity     243.949984   51.467631 -192.482353
                  F-Gases          42.544553   26.330346  -16.214207
                  Industry        221.311343  133.396791  -87.914552
                  Others            0.106952    0.267494    0.160542
                  Transportation  118.196168   89.348627  -28.847540
                  Waste            15.363250   10.853253   -4.509998

In [183]:
emiss_2018 = 783.8
emiss_2020 = 670.56
power_base = emiss_2020
power_ep = 	-192.48
power_cp = -125.61
ind_base = emiss_2020 + power_ep
ind_ep = -87.91
ind_cp = -16.05
trn_base = ind_base + ind_ep
trn_ep = -28.84
trn_cp = -18.13
bld_base = trn_base + trn_ep
bld_ep = -15.70
bld_cp = -2.93
waste_base = bld_base + bld_ep
waste_ep = -4.50
waste_cp = -2.19
fgas_base = waste_base + waste_ep
fgas_ep = -16.39
fgas_cp = -7.08

dac_base = fgas_base + fgas_ep
dac_ep = -5.91
dac_cp = 0

lulucf_ep = -9
lulucf_cp = 0
lulucf_base = dac_base + dac_ep

others_ep = -0.98+0.15
others_cp = 0
others_base = lulucf_base + lulucf_ep

emiss_2035_ep = 309.33


# power_2018 = 278.85
# ind_2018 = 279.97
# trn_2018 = 98.76
# bld_2018 = 48.90
# waste_2018 = 19.3
# fgas_2018 = 23.59

power_2020 = 243.29# * 1.221
ind_2020 = 	205.19# * 1.069
trn_2020 = 118.20# * 1.020
bld_2020 = 50.25# * 1.120
waste_2020 = 15.36# * 1.047
fgas_2020 = 42.73# * 0.870


rr_power = (1 - (1 + power_ep / power_2020) * (1 - 0.181)) * 100
rr_ind = (1 - (1+ ind_ep / ind_2020) * (1 - 0.040)) * 100
rr_trn = (1 - (1+trn_ep / trn_2020) * (1 - 0.020)) * 100
rr_bld = (1 - (1+bld_ep / bld_2020) * (1 - 0.107)) * 100
rr_waste = (1 - (1+waste_ep / waste_2020) * (1 - 0.045)) * 100
rr_fgas = (1 - (1+fgas_ep / fgas_2020) * (1  + 0.150)) * 100

In [184]:
ind_base

478.0799999999999

In [185]:
emiss_2035_ep - emiss_2018

-474.46999999999997

In [186]:
rd_ttl = emiss_2035_ep - emiss_2018
rr_ttl = -rd_ttl / emiss_2018 * 100

In [187]:
rr_ttl

60.5345751467211

In [192]:
data = pd.DataFrame({
    "category": [
        "2018",
        "2020",
        "Power", "Industry", "Transportation", "Buildings", "Waste", "F-Gases", "DAC", "LULUCF", "Others", 
        "2035"
    ],
})

fig = go.Figure()

# Start and end bars
fig.add_trace(go.Waterfall(
    name="Enhanced Ambition",
    orientation="v",
    measure=["absolute"] * 2 + ["relative"] * 9 + ["total"],
    x=data["category"],
    y=[emiss_2018, emiss_2020, power_ep, ind_ep, trn_ep, bld_ep, waste_ep, fgas_ep, dac_ep, lulucf_ep, others_ep, emiss_2035_ep],
    base=0,
    connector={"visible": False},
    decreasing={"marker": {"color": "#1f77b4"}},   # enhanced ambition
    increasing={"marker": {"color": "#FF6692"}},   # current policies
    totals={"marker": {"color": "lightgray"}},
    showlegend=False
))


# Add Current Policy overlays just for Coal categories
fig.add_trace(go.Bar(
    name="Current Policy",
    x=["Power", "Industry", "Transportation", "Buildings", "Waste", "F-Gases", "DAC", "LULUCF", "Others"],
    y=[power_cp, ind_cp, trn_cp, bld_cp, waste_cp, fgas_cp, dac_cp, lulucf_cp, others_cp],  # smaller reductions
    base=[power_base, ind_base, trn_base, bld_base, waste_base, fgas_base, dac_base, lulucf_base, others_base],  # position on top of previous waterfall step
    marker_color="#AEC7E8",
    showlegend=False
))

fig.update_layout(
    width=1050,
    height=500,
    font=dict(size=12),
    plot_bgcolor='white',
    paper_bgcolor='white',
    # title="<b>Emissions Reductions from Each Sector Compared to 2018 Levels</b>",
    # title_font_size=21,
    # title_x=0.5,

)

# Add dummy scatter trace to label bar values at center
fig.add_trace(go.Scatter(
    x=["2018", "2020", "2035"],
    y=[emiss_2020 / 2, emiss_2020 / 2, emiss_2035_ep / 2],
    mode="text",
    text=[f"<b>{emiss_2018:.1f}<br>TOTAL</b>", f"<b>{emiss_2020:.1f}<br>NET</b>", f"<b>{emiss_2035_ep:.1f}<br>NET</b>"],
    textposition="middle center",
    showlegend=False
))

# Add dummy scatter trace to label bar values at center
fig.add_trace(go.Scatter(
    x=["Power", "Industry", "Transportation", "Buildings", "Waste", "F-Gases", "DAC", "LULUCF", 'Others'],
    #-190.5, -73.8, -28.9, -15.9, -4.5, -4.9, -3.0, -9
    y=[x+40 for x in [power_base, ind_base, trn_base, bld_base, waste_base, fgas_base]] + [y + 10 for y in [dac_base, lulucf_base, others_base]],
    mode="text",
    text=[f"<b>{power_ep:.1f}<br>(△{rr_power:.1f}%)</b>", f"<b>{ind_ep:.1f}<br>(△{rr_ind:.1f}%)</b>", f"<b>{trn_ep:.1f}<br>(△{rr_trn:.1f}%)</b>", 
          f"<b>{bld_ep:.1f}<br>(△{rr_bld:.1f}%)</b>", f"<b>{waste_ep:.1f}<br>(△{rr_waste:.1f}%)</b>", f"<b>{fgas_ep:.1f}<br>(△{rr_fgas:.1f}%)</b>",
          f"<b>{dac_ep:.1f}</b>", f"<b>{lulucf_ep:.1f}</b>", f"<b>{others_ep:.1f}</b>"],
    textposition="middle center",
    showlegend=False
))

fig.add_trace(go.Bar(
    x=[None], y=[None],
    name="Current Policy",
    marker=dict(color="#AEC7E8"),
    showlegend=True,
    hoverinfo="skip"
))

fig.add_trace(go.Bar(
    x=[None], y=[None],
    name="Enhanced Ambition",
    marker=dict(color="#1f77b4"),
    showlegend=True,
    hoverinfo="skip"
))


fig.update_layout(
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=-0.3,
        xanchor='center',
        x=0.5,
        bgcolor='rgba(0,0,0,0)',
        borderwidth=0,
        font=dict(size=15)
    )
)

fig.update_layout(
    yaxis=dict(
        showgrid=True,
        gridcolor='lightgrey',
        title="Emission (MtCO2e)", title_font_size=18,
        tickvals=list(range(0, 801, 100)),
    )
)

fig.add_annotation(
    x=11, y=emiss_2035_ep + 5,        # 7 is the index of "2035" in the x-category list
    ax=11, ay=emiss_2018 + 30,
    xref="x", yref="y",
    axref="x", ayref="y",
    text=f"<b>{rd_ttl:.1f}<br>(△{rr_ttl:.1f}%)</b>",
    showarrow=True,
    arrowhead=3,
    arrowwidth=3,
    arrowsize=1,
    arrowcolor="#1f77b4",
    font=dict(size=12, color="black"),
    align="center"
)

fig.add_annotation(
    x=2 + 0.2, y=ind_base + 5,        # 7 is the index of "2035" in the x-category list
    ax=2 + 0.2, ay=emiss_2020,
    xref="x", yref="y",
    axref="x", ayref="y",
    # text="<b>Enhanced<br>Ambition</b>",
    showarrow=True,
    arrowhead=1,
    arrowwidth=3,
    arrowsize=1,
    arrowcolor="#00A08B",
    font=dict(size=10, color="#00A08B"),
    align="center"
)

fig.add_annotation(
    x=2 -0.2, y=emiss_2020 + power_cp + 5,        # 7 is the index of "2035" in the x-category list
    ax=2 - 0.2, ay=emiss_2020,
    xref="x", yref="y",
    axref="x", ayref="y",
    # text="<b>CoalOut</b>",
    showarrow=True,
    arrowhead=1,
    arrowwidth=3,
    arrowsize=1,
    arrowcolor="#1616A7",
    font=dict(size=10, color="#1616A7"),
    align="center"
)


fig.update_layout(
    xaxis=dict(
        tickvals=list(range(len(data['category']))),
        ticktext=[f"<b>{cat}</b>" for cat in data['category']],
        # tickfont=dict(size=15)
    )
)

fig.add_annotation(
    text="<b>Current<br>Policy</b>",
    # xref="paper", yref="paper",
    x=2-0.6, y=620,
    showarrow=False,
    font=dict(size=10, color="#1616A7"),
    align="left"
)

fig.add_annotation(
    text="<b>Enhanced<br>Ambition</b>",
    # xref="paper", yref="paper",
    x=2+0.65, y=620,
    showarrow=False,
    font=dict(size=10, color="#00A08B"),
    align="left"
)

fig